# Lecture 7–9 — Practice: Detection → Error Analysis → Segmentation

Это **практика без оценивания**.

Что будет:
1) IoU и NMS на синтетических bbox (без интернета)  
2) Ошибки детекции: TP/FP/FN/дубликаты/локализация  
3) Быстрый YOLO baseline (опционально, если есть интернет)  
4) Сегментация на синтетических масках + метрики IoU/Dice

В ноутбуке есть офлайн-режим, который работает без скачиваний.


In [ ]:
# Install deps
!pip -q install numpy matplotlib scikit-learn opencv-python-headless
# Optional for YOLO (internet needed):
!pip -q install ultralytics || true


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams["figure.dpi"] = 140

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
seed_everything(42)


## 1) IoU для bbox

IoU = площадь пересечения / площадь объединения.

In [ ]:
def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    iw = max(0.0, inter_x2 - inter_x1)
    ih = max(0.0, inter_y2 - inter_y1)
    inter = iw * ih
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    union = area_a + area_b - inter
    return 0.0 if union <= 0 else inter / union

A = (10, 10, 60, 60)
B = (30, 30, 80, 80)
print("IoU(A,B) =", round(iou_xyxy(A,B), 4))


## 2) NMS (Non-Maximum Suppression)

Оставляем лучшую рамку, убираем близкие по IoU.

In [ ]:
def nms(boxes, scores, iou_thr=0.5):
    idxs = np.argsort(scores)[::-1].tolist()
    keep = []
    while idxs:
        i = idxs.pop(0)
        keep.append(i)
        rest = []
        for j in idxs:
            if iou_xyxy(boxes[i], boxes[j]) < iou_thr:
                rest.append(j)
        idxs = rest
    return keep

boxes = [(10,10,60,60),(12,12,58,58),(70,10,120,60)]
scores = [0.9, 0.8, 0.7]
print("keep idx:", nms(boxes, scores, iou_thr=0.5))


## 3) Ошибки детекции: TP/FP/FN и matching

Сопоставляем предсказания с GT по IoU.

In [ ]:
def match_predictions(gt_boxes, pred_boxes, pred_scores, iou_thr=0.5):
    order = np.argsort(pred_scores)[::-1]
    gt_used = [False]*len(gt_boxes)
    matches = []
    for pi in order:
        best_iou = 0.0
        best_gi = None
        for gi, g in enumerate(gt_boxes):
            if gt_used[gi]:
                continue
            iou = iou_xyxy(pred_boxes[pi], g)
            if iou > best_iou:
                best_iou = iou
                best_gi = gi
        if best_gi is not None and best_iou >= iou_thr:
            gt_used[best_gi] = True
            matches.append((int(pi), int(best_gi), float(best_iou)))
        else:
            matches.append((int(pi), None, float(best_iou)))
    tp = sum(1 for _,gi,_ in matches if gi is not None)
    fp = sum(1 for _,gi,_ in matches if gi is None)
    fn = sum(1 for u in gt_used if not u)
    return tp, fp, fn, matches

gt = [(10,10,60,60), (70,10,120,60)]
pred = [(12,12,58,58), (14,14,56,56), (72,12,118,58), (140,10,170,40)]
scores = [0.9, 0.6, 0.8, 0.7]

tp, fp, fn, matches = match_predictions(gt, pred, scores, iou_thr=0.5)
print("TP,FP,FN =", tp, fp, fn)
print("matches:", matches)


## 4) PR-кривая (игрушечная)

Меняем порог confidence и смотрим Precision/Recall.

In [ ]:
def pr_curve_from_scores(gt_boxes, pred_boxes, pred_scores, iou_thr=0.5, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0, 1, 51)
    precs, recs = [], []
    for t in thresholds:
        keep = [i for i,s in enumerate(pred_scores) if s >= t]
        pb = [pred_boxes[i] for i in keep]
        ps = [pred_scores[i] for i in keep]
        if keep:
            tp, fp, fn, _ = match_predictions(gt_boxes, pb, ps, iou_thr=iou_thr)
        else:
            tp, fp, fn = 0, 0, len(gt_boxes)
        prec = tp / max(1, tp+fp)
        rec = tp / max(1, tp+fn)
        precs.append(prec); recs.append(rec)
    return np.array(precs), np.array(recs), np.array(thresholds)

gt = [(10,10,60,60), (70,10,120,60)]
pred = [(12,12,58,58), (14,14,56,56), (72,12,118,58), (140,10,170,40)]
scores = [0.9, 0.6, 0.8, 0.7]

P,R,T = pr_curve_from_scores(gt, pred, scores, iou_thr=0.5)
plt.plot(R, P)
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("PR curve (toy detector)")
plt.grid(True); plt.show()


## 5) YOLO baseline (опционально)

Если интернет есть — попробуйте готовую модель.

In [ ]:
USE_YOLO = False  # переключи на True, если хочешь попробовать

if USE_YOLO:
    from ultralytics import YOLO
    import requests
    from PIL import Image
    from io import BytesIO

    url = "https://ultralytics.com/images/bus.jpg"
    img = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

    model = YOLO("yolov8n.pt")
    res = model(img, verbose=False)[0]
    print("detections:", len(res.boxes))
    im = res.plot()
    plt.figure(figsize=(10,6))
    plt.imshow(im); plt.axis("off"); plt.show()
else:
    print("YOLO section skipped (offline mode).")


## 6) Сегментация: IoU/Dice на масках

GT маска и «предсказание» с шумом.

In [ ]:
def make_mask(h=128, w=128, seed=0):
    rng = np.random.RandomState(seed)
    mask = np.zeros((h,w), dtype=np.uint8)
    cx, cy = int(rng.uniform(w*0.3, w*0.7)), int(rng.uniform(h*0.3, h*0.7))
    r = int(rng.uniform(min(h,w)*0.12, min(h,w)*0.22))
    cv2.circle(mask, (cx,cy), r, 1, -1)
    return mask

def iou_mask(gt, pr):
    gt = gt.astype(bool); pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum()
    union = np.logical_or(gt, pr).sum()
    return 0.0 if union == 0 else inter / union

def dice_mask(gt, pr):
    gt = gt.astype(bool); pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum()
    denom = gt.sum() + pr.sum()
    return 1.0 if denom == 0 else (2.0*inter) / denom

gt = make_mask(seed=1)
pr = cv2.dilate(gt.copy(), np.ones((3,3), np.uint8), iterations=1)
noise = (np.random.rand(*pr.shape) < 0.01).astype(np.uint8)
pr = np.clip(pr + noise, 0, 1)

print("IoU:", round(iou_mask(gt, pr), 4), "Dice:", round(dice_mask(gt, pr), 4))

plt.figure(figsize=(9,3))
plt.subplot(1,3,1); plt.title("GT"); plt.imshow(gt, cmap="gray"); plt.axis("off")
plt.subplot(1,3,2); plt.title("Pred"); plt.imshow(pr, cmap="gray"); plt.axis("off")
plt.subplot(1,3,3); plt.title("Error"); plt.imshow((gt!=pr).astype(np.uint8), cmap="gray"); plt.axis("off")
plt.tight_layout(); plt.show()
